In [4]:
import planetary_computer
import itertools
import dask.dataframe as dd

cc = planetary_computer.get_container_client("pcstacitems", "items")

blobs = list(cc.list_blobs("landsat-c2-l2.parquet/"))

def key(blob):
    return blob.name.split("/")[1].split("_")[0]

keep_blobs = []
for k, v in itertools.groupby(sorted(blobs, key=key), key=key):
    v = list(v)
    blob = max(v, key=lambda x: x.last_modified)
    keep_blobs.append(blob)
    
uris = [f"az://items/{blob.name}" for blob in keep_blobs]

In [6]:
df = dd.read_parquet(uris, storage_options={"account_name": "pcstacitems", "credential": planetary_computer.sas.get_token("pcstacitems", "items").token})
df.head()

,assets,bbox,collection,geometry,id,links,stac_extensions,stac_version,type,created,...,landsat:wrs_row,landsat:wrs_type,platform,proj:epsg,proj:shape,proj:transform,sci:doi,view:off_nadir,view:sun_azimuth,view:sun_elevation
0,{'ang': {'description': 'Collection 2 Level-1 ...,"{'xmin': -110.86815155, 'ymin': 42.16322498, '...",landsat-c2-l2,b'\x01\x03\x00\x00\x00\x01\x00\x00\x00\x05\x00...,LT04_L2SP_037030_19821230_02_T1,[{'href': 'https://planetarycomputer.microsoft...,[https://stac-extensions.github.io/raster/v1.0...,1.0.0,Feature,2022-05-06 17:02:37.954316+00:00,...,030,2,landsat-4,32612,"[7081, 7831]","[30.0, 0.0, 510885.0, 0.0, -30.0, 4884615.0]",10.5066/P9IAXOVV,0,153.193181,18.969958
1,{'ang': {'description': 'Collection 2 Level-1 ...,"{'xmin': -111.39839774, 'ymin': 40.78781495, '...",landsat-c2-l2,"b""\x01\x03\x00\x00\x00\x01\x00\x00\x00\x05\x00...",LT04_L2SP_037031_19821128_02_T1,[{'href': 'https://planetarycomputer.microsoft...,[https://stac-extensions.github.io/raster/v1.0...,1.0.0,Feature,2022-05-06 17:02:38.114012+00:00,...,031,2,landsat-4,32612,"[7101, 7851]","[30.0, 0.0, 467385.0, 0.0, -30.0, 4731015.0]",10.5066/P9IAXOVV,0,155.242596,22.950310
2,{'ang': {'description': 'Collection 2 Level-1 ...,"{'xmin': -111.33233769, 'ymin': 40.76756495, '...",landsat-c2-l2,b'\x01\x03\x00\x00\x00\x01\x00\x00\x00\x05\x00...,LT04_L2SP_037031_19821214_02_T1,[{'href': 'https://planetarycomputer.microsoft...,[https://stac-extensions.github.io/raster/v1.0...,1.0.0,Feature,2022-05-06 17:02:38.293544+00:00,...,031,2,landsat-4,32612,"[7101, 7851]","[30.0, 0.0, 472785.0, 0.0, -30.0, 4728915.0]",10.5066/P9IAXOVV,0,154.407765,20.676323
3,{'ang': {'description': 'Collection 2 Level-1 ...,"{'xmin': -111.3140177, 'ymin': 40.76995495, 'x...",landsat-c2-l2,b'\x01\x03\x00\x00\x00\x01\x00\x00\x00\x05\x00...,LT04_L2SP_037031_19821230_02_T2,[{'href': 'https://planetarycomputer.microsoft...,[https://stac-extensions.github.io/raster/v1.0...,1.0.0,Feature,2022-05-06 17:02:38.451790+00:00,...,031,2,landsat-4,32612,"[7091, 7841]","[30.0, 0.0, 474285.0, 0.0, -30.0, 4728915.0]",10.5066/P9IAXOVV,0,152.624197,20.101315
4,{'ang': {'description': 'Collection 2 Level-1 ...,"{'xmin': -71.2485277, 'ymin': 40.77333499, 'xm...",landsat-c2-l2,b'\x01\x03\x00\x00\x00\x01\x00\x00\x00\x05\x00...,LT04_L2SP_011031_19821208_02_T2,[{'href': 'https://planetarycomputer.microsoft...,[https://stac-extensions.github.io/raster/v1.0...,1.0.0,Feature,2022-05-06 16:49:18.334263+00:00,...,031,2,landsat-4,32619,"[7201, 7951]","[30.0, 0.0, 315885.0, 0.0, -30.0, 4731915.0]",10.5066/P9IAXOVV,0,154.862849,21.352277


In [7]:
df.columns

Index(['assets', 'bbox', 'collection', 'geometry', 'id', 'links',
       'stac_extensions', 'stac_version', 'type', 'created', 'datetime',
       'description', 'eo:cloud_cover', 'gsd', 'instruments',
       'landsat:cloud_cover_land', 'landsat:collection_category',
       'landsat:collection_number', 'landsat:correction', 'landsat:scene_id',
       'landsat:wrs_path', 'landsat:wrs_row', 'landsat:wrs_type', 'platform',
       'proj:epsg', 'proj:shape', 'proj:transform', 'sci:doi',
       'view:off_nadir', 'view:sun_azimuth', 'view:sun_elevation'],
      dtype='object')

In [10]:
# df = df[['id', 'geometry', 'bbox', 'datetime', 'eo:cloud_cover', "s2:product_uri", "s2:granule_id", "s2:nodata_pixel_percentage", "s2:saturated_defective_pixel_percentage"]]
df = df[['id', 'geometry', 'datetime', 'eo:cloud_cover', "landsat:scene_id", "proj:epsg", "proj:shape", "proj:transform", "landsat:collection_category", "landsat:collection_number", "landsat:correction", "sci:doi"]]

df["datetime"] = dd.to_datetime(df["datetime"])

In [11]:
df.head()

,id,geometry,datetime,eo:cloud_cover,landsat:scene_id,proj:epsg,proj:shape,proj:transform,landsat:collection_category,landsat:collection_number,landsat:correction,sci:doi
0,LT04_L2SP_037030_19821230_02_T1,b'\x01\x03\x00\x00\x00\x01\x00\x00\x00\x05\x00...,1982-12-30 17:29:45.886000+00:00,8.0,LT40370301982364XXX01,32612,"[7081, 7831]","[30.0, 0.0, 510885.0, 0.0, -30.0, 4884615.0]",T1,02,L2SP,10.5066/P9IAXOVV
1,LT04_L2SP_037031_19821128_02_T1,"b""\x01\x03\x00\x00\x00\x01\x00\x00\x00\x05\x00...",1982-11-28 17:30:07.254044+00:00,48.0,LT40370311982332XXX01,32612,"[7101, 7851]","[30.0, 0.0, 467385.0, 0.0, -30.0, 4731015.0]",T1,02,L2SP,10.5066/P9IAXOVV
2,LT04_L2SP_037031_19821214_02_T1,b'\x01\x03\x00\x00\x00\x01\x00\x00\x00\x05\x00...,1982-12-14 17:30:01.944025+00:00,47.0,LT40370311982348XXX01,32612,"[7101, 7851]","[30.0, 0.0, 472785.0, 0.0, -30.0, 4728915.0]",T1,02,L2SP,10.5066/P9IAXOVV
3,LT04_L2SP_037031_19821230_02_T2,b'\x01\x03\x00\x00\x00\x01\x00\x00\x00\x05\x00...,1982-12-30 17:30:09.568025+00:00,1.0,LT40370311982364XXX05,32612,"[7091, 7841]","[30.0, 0.0, 474285.0, 0.0, -30.0, 4728915.0]",T2,02,L2SP,10.5066/P9IAXOVV
4,LT04_L2SP_011031_19821208_02_T2,b'\x01\x03\x00\x00\x00\x01\x00\x00\x00\x05\x00...,1982-12-08 14:49:27.753038+00:00,5.0,LT40110311982342PAC00,32619,"[7201, 7951]","[30.0, 0.0, 315885.0, 0.0, -30.0, 4731915.0]",T2,02,L2SP,10.5066/P9IAXOVV


In [12]:
filtered_df = df[(df["datetime"] >= "1982-01-01") &
                 (df["eo:cloud_cover"] < 20)]

In [13]:
filtered_df = filtered_df.compute()
print(len(filtered_df))

4145207


In [14]:
filtered_df.head()

,id,geometry,datetime,eo:cloud_cover,landsat:scene_id,proj:epsg,proj:shape,proj:transform,landsat:collection_category,landsat:collection_number,landsat:correction,sci:doi
0,LT04_L2SP_037030_19821230_02_T1,b'\x01\x03\x00\x00\x00\x01\x00\x00\x00\x05\x00...,1982-12-30 17:29:45.886000+00:00,8.0,LT40370301982364XXX01,32612,"[7081, 7831]","[30.0, 0.0, 510885.0, 0.0, -30.0, 4884615.0]",T1,02,L2SP,10.5066/P9IAXOVV
3,LT04_L2SP_037031_19821230_02_T2,b'\x01\x03\x00\x00\x00\x01\x00\x00\x00\x05\x00...,1982-12-30 17:30:09.568025+00:00,1.0,LT40370311982364XXX05,32612,"[7091, 7841]","[30.0, 0.0, 474285.0, 0.0, -30.0, 4728915.0]",T2,02,L2SP,10.5066/P9IAXOVV
4,LT04_L2SP_011031_19821208_02_T2,b'\x01\x03\x00\x00\x00\x01\x00\x00\x00\x05\x00...,1982-12-08 14:49:27.753038+00:00,5.0,LT40110311982342PAC00,32619,"[7201, 7951]","[30.0, 0.0, 315885.0, 0.0, -30.0, 4731915.0]",T2,02,L2SP,10.5066/P9IAXOVV
10,LT04_L2SP_041026_19821210_02_T2,b'\x01\x03\x00\x00\x00\x01\x00\x00\x00\x05\x00...,1982-12-10 17:52:51.504038+00:00,2.0,LT40410261982344XXX04,32612,"[7451, 8151]","[30.0, 0.0, 192585.0, 0.0, -30.0, 5526015.0]",T2,02,L2SP,10.5066/P9IAXOVV
14,LT04_L2SP_041028_19821124_02_T1,b'\x01\x03\x00\x00\x00\x01\x00\x00\x00\x05\x00...,1982-11-24 17:53:34.799006+00:00,0.0,LT40410281982328XXX05,32611,"[7091, 7821]","[30.0, 0.0, 563985.0, 0.0, -30.0, 5207115.0]",T1,02,L2SP,10.5066/P9IAXOVV


In [15]:
max_date = filtered_df['datetime'].max().strftime("%Y_%m_%d")
print(max_date)

2026_08_08


In [ ]:
filtered_df.to_parquet(f"landsat_lt_{max_date}_gt_1982_01_01.parquet", index=False)
print(f"Saved to: landsat_lt_{max_date}_gt_1982_01_01.parquet")